In [1]:
from go_bpe import BPETokenizer


In [2]:
with open("./the-verdict.txt", "r", encoding="utf-8") as f:
    verdict_corpus = f.read()


In [15]:
vocab_size = 65536
output_dim = 256

In [ ]:
tok = BPETokenizer.train(verdict_corpus, max_vocab_size=vocab_size)
tok.save("model.json")

In [4]:
with BPETokenizer.load("model.json") as tok:
    ids = tok.encode(verdict_corpus)
    text = tok.decode(ids)

In [ ]:
print(text)

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

class JohnnyDataset(Dataset):
    def __init__(self, txt, tok, context_size, shift):
        self.input_ids = []
        self.target_ids = []
        token_ids = tok.encode(txt)

        for i in range(0, len(token_ids) - context_size, shift):
            input_id = token_ids[i : i + context_size]
            target_id = token_ids[i + 1 : i + context_size + 1]
            self.input_ids.append(torch.tensor(input_id))
            self.target_ids.append(torch.tensor(target_id))

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.target_ids)


In [13]:
def create_dataloader(
    txt,
    vocab_json="model.json",
    context_size=256,
    shift=128,
    batch_size=4,
    drop_last=True,
    num_workers=0,
    shuffle=True,
):
    tok = BPETokenizer.load(vocab_json)
    dataset = JohnnyDataset(txt, tok, context_size, shift)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        drop_last=drop_last,
        num_workers=num_workers,
        shuffle=shuffle,
    )

    return dataloader

In [31]:
context_size = 128
dataloader = create_dataloader(verdict_corpus, context_size=context_size, shift=128, batch_size=8)
data_iter = iter(dataloader)


In [32]:
token_emb_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_emb_layer = torch.nn.Embedding(context_size, output_dim)
inputs, outputs = next(data_iter)

In [33]:
token_embs = token_emb_layer(inputs)
token_embs.shape

torch.Size([8, 128, 256])

In [38]:
pos_embs = pos_emb_layer(torch.arange(context_size))
pos_embs.shape

torch.Size([128, 256])

In [39]:
(token_embs + pos_embs).shape

torch.Size([8, 128, 256])

In [ ]:

class MHSA(torch.nn.Module):
    mask: torch.Tensor

    def __init__(self, d_in, d_out, context_size, heads_len, dropout, bias=False):
        super().__init__()
        self.head_out = d_out // heads_len # 8 / 2 = 4
        self.heads_len = heads_len
        self.d_out = d_out
        self.qW = torch.nn.Linear(d_in, d_out, bias=bias) # 3, 8
        self.kW = torch.nn.Linear(d_in, d_out, bias=bias)
        self.vW = torch.nn.Linear(d_in, d_out, bias=bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_size, context_size), diagonal=1)
)

    def forward(self, x):
        b, context_size, d_in = x.shape # 2, 4, 3
        queries = self.qW(x)
        keys = self.kW(x)
        values = self.vW(x) # 2, 4, 8

        queries = queries.view(b, context_size, self.heads_len, self.head_out)
        keys = keys.view(b, context_size, self.heads_len, self.head_out)
        values = values.view(b, context_size, self.heads_len, self.head_out)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2) # 2,2,3,4

        attention_scores = queries @ keys.transpose(2, 3) # 2,2,3,4 @ 2,2,4,3 = 2,2,3,3
        mask = self.mask.bool()[:context_size, :context_size]
        attention_scores.masked_fill_(mask, -torch.inf)
        attention_w = torch.softmax(attention_scores / keys.shape[-1]**0.5, dim=-1)

        attention_w = self.dropout(attention_w)
        context = attention_w @ values # 2,2,3,4

        context = context.transpose(1,2)
        context = context.view(b, context_size, self.d_out)

        return context






AttributeError: module 'torch.nn' has no attribute 'Layer'